# Uygulama 1 — Adli Laboratuvarda Mitokondriyal DNA

**Hafta 1** — Dizgi işlemleri, baz sayımı, GC oranı, ters tamamlayıcı, motif beklentisi

**Veri:** NC_012920.1 — İnsan mitokondriyal genomu (16 569 bp)

---

**Senaryo:** Bir adli tıp laboratuvarında stajyersin. Eski kemik örneklerinde çekirdek DNA'sı çoğu zaman bozulmuş olduğu için kimlik tespitinde mitokondriyal DNA (mtDNA) kullanılır. Laboratuvar sorumlusu senden, referans olarak kullanılan insan mtDNA dizisini (NC_012920.1) NCBI'dan indirip temel özelliklerini çıkarmanı istiyor.

## Kalem-Kağıt Isınması

Örnek tüpünün etiketinde mtDNA'dan alınmış `5'-GATCACAGGT-3'` dizisi yazıyor.

**(a) Ters tamamlayıcısı:**
- Önce her harfi eşiyle değiştir: `CTAGTGTCCA`
- Sonra ters çevir: **`5'-ACCTGTGATC-3'`**

**(b) GC oranı:**
- G=3, C=2 → GC = 5/10 = **%50**

**(c) 6 bazlık motif beklentisi (16 569 bp, eşit olasılık):**
- P = (1/4)⁶ = 1/4096
- Beklenen ≈ 16 569 / 4096 ≈ **4,04 kez**

---
## Görev 0 — Gerekli Kütüphaneler

- **`Bio.Entrez`** → NCBI veritabanıyla konuşmamızı sağlar (veri indirme)
- **`Bio.SeqIO`** → Dizi dosyalarını okuma/yazma (FASTA, GenBank vs.)
- **`Bio.Seq`** → DNA dizisini "akıllı bir string" olarak tutar; `.reverse_complement()`, `.count()` gibi biyolojik işlemler hazır gelir

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from collections import Counter

# NCBI kim olduğunu bilmek ister
Entrez.email = "semra@example.com"

print("Kütüphaneler yüklendi!")

---
## Görev 1 — NC_012920.1 kaydını FASTA olarak indir

**FASTA** en basit dizi formatı:
```
>başlık satırı
ATCGATCGATCG...
```

- `Entrez.efetch()` → NCBI'dan veri çeker
- `SeqIO.read()` → tek kayıt okur, `SeqRecord` nesnesi döner

In [ ]:
handle = Entrez.efetch(db="nucleotide", id="NC_012920.1",
                       rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()

# Yerel dosyaya kaydet (bir daha indirmemek için)
SeqIO.write(record, "NC_012920.fasta", "fasta")

print(f"Kayıt ID'si  : {record.id}")
print(f"Açıklama      : {record.description}")
print(f"Dizi uzunluğu : {len(record.seq)} bp")

**Beklenen çıktı:** ID = `NC_012920.1`, uzunluk = `16569 bp`

> 💡 Bir sonraki çalıştırmada NCBI'dan tekrar indirmek yerine yerel dosyadan okuyabilirsin:
> ```python
> record = SeqIO.read("NC_012920.fasta", "fasta")
> ```

---
## Görev 2 — A, C, G, T sayıları + belirsiz bazlar

DNA'da normalde 4 harf var: **A, C, G, T**

Ama bazen bir pozisyondaki baz belirlenemez → **N** (= "bilmiyorum") ya da diğer IUPAC kodları (R, Y, W...) kullanılır.

`Counter` ile her harfi sayıyoruz.

In [ ]:
seq = record.seq.upper()
sayimlar = Counter(str(seq))

for baz, sayi in sorted(sayimlar.items()):
    print(f"  {baz} : {sayi}")

# Standart dışı karakter var mı?
standart = {'A', 'C', 'G', 'T'}
belirsiz = {k: v for k, v in sayimlar.items() if k not in standart}

if belirsiz:
    print(f"\n  Standart dışı karakter(ler): {belirsiz}")
    for baz in belirsiz:
        pozlar = [i + 1 for i, c in enumerate(str(seq)) if c == baz]
        print(f"    '{baz}' pozisyonları (1-tabanlı): {pozlar}")
else:
    print("\n  Dizide sadece A, C, G, T var.")

**Beklenen:** 1 adet **N** bazı, pozisyon **3107**. Bu pozisyondaki baz referans dizisinde kasıtlı olarak belirsiz bırakılmış (tarihsel numaralama uyumu için).

---
## Görev 3 — GC Oranı

**GC oranı neden önemli?**
- G–C bağı **3** hidrojen bağı (güçlü), A–T bağı **2** (zayıf)
- Yüksek GC → daha stabil DNA, daha yüksek erime sıcaklığı
- Belirsiz bazları (N) paydadan çıkararak hesaplıyoruz

In [ ]:
g = sayimlar.get('G', 0)
c = sayimlar.get('C', 0)
a = sayimlar.get('A', 0)
t = sayimlar.get('T', 0)
toplam_kesin = a + c + g + t

gc_oran = (g + c) / toplam_kesin * 100

print(f"A={a}, C={c}, G={g}, T={t}")
print(f"Kesin baz toplamı: {toplam_kesin}")
print(f"GC oranı: {gc_oran:.2f}%")
print(f"\nG ({g}) ile C ({c}) sayıları neden farklı?")
print(f"→ FASTA'da sadece TEK iplik var (ileri iplik).")
print(f"→ Chargaff kuralı (A=T, G=C) ÇİFT iplikli DNA için geçerli.")
print(f"→ Tek iplikte G ve C eşit olmak zorunda değil.")

**Beklenen:** GC oranı ≈ **%44.36** — İnsan mtDNA'sı AT-zengin.

---
## Görev 4 — Kesim Enzimi Tanıma Dizileri

**Kesim enzimleri** (restriksiyon enzimleri) DNA'yı belirli dizilerde keser. Düşün: DNA bir ip, enzim belirli bir düğümü tanıyan bir makas.

| Enzim | Tanıma Dizisi |
|-------|---------------|
| EcoRI | GAATTC |
| BamHI | GGATCC |
| HindIII | AAGCTT |

**Beklenen sayı:** `(N - k + 1) × (1/4)^k`  (bazlar eşit dağılımlıysa)

In [ ]:
enzimler = {
    "EcoRI":   "GAATTC",
    "BamHI":   "GGATCC",
    "HindIII": "AAGCTT",
}

seq_str = str(seq)
n = len(seq_str)
k = 6
beklenen = (n - k + 1) * (1 / 4) ** k

print(f"Beklenen (eşit olasılık, k={k}): {beklenen:.2f}\n")

for enzim, motif in enzimler.items():
    pozisyonlar = []
    bas = 0
    while True:
        idx = seq_str.find(motif, bas)
        if idx == -1:
            break
        pozisyonlar.append(idx + 1)  # 1-tabanlı
        bas = idx + 1

    print(f"{enzim} ({motif}):")
    print(f"  Gözlenen: {len(pozisyonlar)} kez")
    print(f"  Pozisyonlar: {pozisyonlar}\n")

**Beklenen:** ~4 kez. EcoRI ve HindIII 3'er kez, BamHI sadece 1 kez bulunuyor — beklentiye yakın.

---
## Görev 5 — GATC Motifi: Beklenti vs Gözlem

GATC motifi (4 baz) DNA metilasyonunda önemli bir sinyal dizisi.

Beklenen sayıyı iki yolla hesaplıyoruz:
1. **Eşit olasılık:** Her baz 1/4 → P = (1/4)⁴
2. **Gerçek frekanslar:** P = f(G) × f(A) × f(T) × f(C)

In [ ]:
motif = "GATC"
k = len(motif)

# Gözlenen sayı
gozlenen = 0
bas = 0
while True:
    idx = seq_str.find(motif, bas)
    if idx == -1:
        break
    gozlenen += 1
    bas = idx + 1

# Beklenen 1: eşit olasılık
beklenen_esit = (n - k + 1) * (1 / 4) ** k

# Beklenen 2: gerçek baz frekansları
fa = a / toplam_kesin
fc = c / toplam_kesin
fg = g / toplam_kesin
ft = t / toplam_kesin

p_gercek = fg * fa * ft * fc  # G × A × T × C
beklenen_gercek = (n - k + 1) * p_gercek

print(f"Gözlenen GATC sayısı     : {gozlenen}")
print(f"Beklenen (eşit olasılık) : {beklenen_esit:.2f}")
print(f"Beklenen (gerçek frekans): {beklenen_gercek:.2f}")
print(f"\nBaz frekansları: A={fa:.4f}  C={fc:.4f}  G={fg:.4f}  T={ft:.4f}")

fark_esit = abs(gozlenen - beklenen_esit)
fark_gercek = abs(gozlenen - beklenen_gercek)
yakin = "Gerçek frekans" if fark_gercek < fark_esit else "Eşit olasılık"
print(f"\nGözleme daha yakın olan: {yakin} tahmini")
print(f"\nAma ikisi de gözlemden çok yüksek!")
print(f"→ GATC motifi mtDNA'da BEKLENTİDEN AZ var (baskılanmış).")

**Yorum:** Gözlenen (23) beklentiden (~52-65) çok az. Bu, mtDNA'da GATC motifinin evrimsel olarak baskılandığını gösteriyor.

---
## Görev 6 — Depolama Boyutu

DNA'da sadece 4 harf var → her baz **2 bit** ile kodlanabilir:
- A=00, C=01, G=10, T=11

| Yöntem | Formül |
|--------|--------|
| Düz metin | 1 byte/baz (8 bit, israf) |
| 2-bit kodlama | 2 bit/baz → 4 baz = 1 byte |

In [ ]:
duz_metin = n * 1              # 1 byte/baz
iki_bit = (n * 2 + 7) // 8     # 2 bit/baz, byte'a yuvarla

print(f"Düz metin (1 byte/baz): {duz_metin:,} byte = {duz_metin/1024:.1f} KB")
print(f"2-bit kodlama:          {iki_bit:,} byte  = {iki_bit/1024:.1f} KB")
print(f"Tasarruf: {(1 - iki_bit/duz_metin)*100:.0f}%")

**Sonuç:** 2-bit kodlama **%75 tasarruf** sağlıyor. İnsan genomunun tamamı (~3 milyar baz) düz metinde ~3 GB iken, 2-bit'te ~750 MB'a düşer.

---

## Uygulama 1 Tamamlandı!

**Öğrendiğin kavramlar:**
- NCBI'dan `Entrez.efetch` ile veri indirme
- `SeqIO.read` ile FASTA okuma
- Baz sayımı (`Counter`), GC oranı hesaplama
- Motif arama (`str.find` döngüsü) ve beklenti hesabı
- Chargaff kuralının tek iplikte neden bozulduğu
- 2-bit DNA kodlama

**Sonraki:** Uygulama 2 — COX1 Geni (transkripsiyon, translasyon, genetik kod tabloları)